# Check from a file, a folder, a .zip all the value. Check if any value is <0


In [1]:
import sys
print(sys.executable)
try:
    import ipywidgets as iw
    print("ipywidgets:", iw.__version__)
except Exception as e:
    print("IMPORT ERROR:", e)

/home/chantel211/Documents/MCTS-Allocator/.venv/bin/python
ipywidgets: 8.1.8


In [2]:
import numpy
import pandas
import matplotlib
import json
import os
import zipfile
from ipywidgets import Dropdown, Button, Output
from IPython.display import display

In [3]:
def gatherDataFromFile(filepath : str) :
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data

def gatherDataFromFolder(folderpath : str) :
    data = []
    for filename in os.listdir(folderpath):
        if filename.endswith('.json'):
            filepath = os.path.join(folderpath, filename)
            data.append(gatherDataFromFile(filepath))
    return data

def gatherDataFromZip(zipfilepath : str) :
    data = []
    with zipfile.ZipFile(zipfilepath, 'r') as zip_file:
        for filename in zip_file.namelist():
            if filename.endswith('.json'):
                with zip_file.open(filename) as file:
                    data.append(json.load(file))
    return data

In [ ]:
import threading, time
from ipywidgets import IntProgress, VBox, Label

# Create a dropdown to select between file, folder, or zip
source_type = Dropdown(
    options=['File', 'Folder', 'Zip'],
    value='File',
    description='Source Type:'
)

# Create a text input for the path
path_input = Dropdown(
    options=[],
    description='Path:'
)

# Create output area
output = Output()

def update_paths(change):
    """Update available paths based on source type"""
    result_path = "../results"
    if not os.path.isdir(result_path):
        result_path = os.getcwd()

    source = source_type.value
    if source == 'File':
        paths = [os.path.join(result_path, f) for f in os.listdir(result_path) if f.endswith('.json')]
    elif source == 'Folder':
        paths = [os.path.join(result_path, f) for f in os.listdir(result_path) if os.path.isdir(os.path.join(result_path, f))]
    else:  # Zip
        paths = [os.path.join(result_path, f) for f in os.listdir(result_path) if f.endswith('.zip')]

    path_input.options = paths

def on_button_click(b):
    """Handle button click to gather and display data"""
    with output:
        output.clear_output()
        path = path_input.value

        if not path:
            print("No path selected.")
            return

        if source_type.value == 'File':
            data = gatherDataFromFile(path)
        elif source_type.value == 'Folder':
            data = gatherDataFromFolder(path)
        else:  # Zip
            data = gatherDataFromZip(path)

        print(f"Data from {source_type.value}: {path}")
        print(json.dumps(data, indent=2))

source_type.observe(update_paths, names='value')


_original_on_button_click = on_button_click

def on_button_click(b):
    progress = IntProgress(min=0, max=100, value=0, description='Loading:')
    label = Label(value='Please wait...')
    container = VBox([progress, label])
    display(container)

    def target():
        try:
            _original_on_button_click(b)
        except Exception as e:
            with output:
                print("Error:", e)

    t = threading.Thread(target=target)
    t.start()

    while t.is_alive():
        progress.value = min(progress.value + 5, 95)
        time.sleep(0.1)

    progress.value = 100
    label.value = 'Done'
    time.sleep(0.2)
    container.close()
update_paths(None)

button = Button(description='Load Data')
button.on_click(on_button_click)

display(source_type, path_input, button, output)

Dropdown(description='Source Type:', options=('File', 'Folder', 'Zip'), value='File')

Dropdown(description='Path:', options=('../results/experiments_28-04-2026_09-37-27.json', '../results/experime…

Button(description='Load Data', style=ButtonStyle())

Output()